# 03 — U-Net++ avec encodeur ResNet18 préentraîné (optimisé L4)

Ce notebook compare **U-Net + ResNet18** et **U-Net++ + ResNet18** avec un protocole identique. Il couvre :

- l'audit des 2 400 paires Water-v2 et la création d'un split fixe par groupe ;
- un prétraitement 512 × 512 avec conservation du ratio et exclusion du padding des métriques ;
- les modèles standard de `segmentation-models-pytorch` avec poids ImageNet ;
- BCE + Dice, AMP, accumulation de gradients, reprise et early stopping ;
- IoU, Dice, précision, rappel, accuracy et Boundary F1, globalement et par groupe ;
- un lancement optionnel sur **Modal L4** avec Volumes persistants.

> Le test final reste fermé tant que l'architecture et le seuil ne sont pas figés. Par défaut, aucune longue expérience et aucune évaluation finale ne sont lancées automatiquement.

## 1. Protocole expérimental

La comparaison change uniquement le décodeur : `smp.Unet` contre `smp.UnetPlusPlus`. L'encodeur, les données, les augmentations, la loss et les règles de sélection restent identiques.

Le split cible contient 14 groupes d'entraînement, 4 de validation et 4 de test. `ADE20K` et `river_segs`, les deux groupes dominants, restent dans l'entraînement. Les 20 autres groupes sont répartis par une recherche déterministe qui équilibre le nombre d'images, la proportion d'eau et la fréquence des masques vides entre validation et test.

In [20]:
%uv pip install "torch==2.12.1" "torchvision==0.27.1" --index-url https://download.pytorch.org/whl/cu126
%uv pip install "segmentation-models-pytorch==0.5.0"
# Si Modal indique qu'un paquet déjà importé a changé, redémarrez le kernel puis reprenez à la cellule suivante.

Using Python 3.12.6 environment at: /usr/local
Audited 2 packages in 20ms
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.6 environment at: /usr/local
Audited 1 package in 17ms
Note: you may need to restart the kernel to use updated packages.


In [21]:
from pathlib import Path

MODAL_VOLUME_ROOT = Path("/mnt/water-segmentation")
if not MODAL_VOLUME_ROOT.exists():
    raise FileNotFoundError(
        "Le Volume Modal attendu n'est pas monté sous /mnt/water-segmentation"
    )
print("Dossier courant :", Path.cwd())
print("Volume projet  :", MODAL_VOLUME_ROOT)
print("Contenu        :", sorted(path.name for path in MODAL_VOLUME_ROOT.iterdir()))

Dossier courant : /root
Volume projet  : /mnt/water-segmentation
Contenu        : ['downloads', 'project']


In [22]:
%uv pip install segmentation-models-pytorch

Using Python 3.12.6 environment at: /usr/local
Audited 1 package in 16ms
Note: you may need to restart the kernel to use updated packages.


In [23]:
from __future__ import annotations

import hashlib
import json
import math
import os
import platform
import random
import subprocess
import sys
import time
from collections import defaultdict
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Any, Literal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset

try:
    import segmentation_models_pytorch as smp
    from torchvision.transforms import InterpolationMode
    from torchvision.transforms import functional as TF
except ImportError as exc:
    raise ImportError(
        "Installez l'environnement avec `uv sync`, puis redémarrez le kernel. "
        "Le notebook requiert segmentation-models-pytorch et torchvision."
    ) from exc


def find_project_root(start: Path) -> Path:
    modal_root = Path("/mnt/water-segmentation/project")
    for candidate in (modal_root, start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Impossible de trouver la racine contenant pyproject.toml")


PROJECT_ROOT = (
    Path(os.environ["PROJECT_ROOT"]).resolve()
    if "PROJECT_ROOT" in os.environ
    else find_project_root(Path.cwd()).resolve()
)

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from water_detection_methods.data import find_water_v2_pairs, load_pair_with_padding
from water_detection_methods.paths import WATER_V2_DIR

DATASET_DIR = Path(os.environ.get("WATER_V2_DIR", WATER_V2_DIR)).resolve()
ARTIFACT_ROOT = Path(
    os.environ.get("ARTIFACT_DIR", PROJECT_ROOT / "runs" / "unetplusplus")
).resolve()
SPLIT_DIR = PROJECT_ROOT / "splits"
STATS_CACHE = ARTIFACT_ROOT / "water_v2_pair_stats.csv"
SPLIT_PATH = SPLIT_DIR / "water_v2_group_split_seed42.json"


@dataclass(frozen=True)
class TrainingConfig:
    image_size: tuple[int, int] = (512, 512)
    batch_size: int = 32
    gradient_accumulation_steps: int = 1
    num_workers: int = 8
    max_epochs: int = 80
    freeze_encoder_epochs: int = 5
    early_stopping_patience: int = 15
    encoder_learning_rate: float = 1e-4
    decoder_learning_rate: float = 3e-4
    weight_decay: float = 1e-4
    threshold: float = 0.5
    boundary_tolerance: int = 2
    seed: int = 42
    use_amp: bool = True


CONFIG = TrainingConfig(
    batch_size=32,
    gradient_accumulation_steps=1,
    num_workers=0 if platform.system() == "Windows" else 8,
)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Laissez `smoke` pour vérifier le pipeline sur quelques images.
# Passez ensuite à `train` pour l'entraînement U-Net++ complet.
##RUN_MODE = os.environ.get("RUN_MODE", "smoke").strip().lower()
RUN_MODE = "train"
if RUN_MODE not in {"inspect", "smoke", "train"}:
    raise ValueError("RUN_MODE doit valoir inspect, smoke ou train")
TRAIN_ARCHITECTURES = ("unetplusplus",)
RUN_ARCHITECTURES = TRAIN_ARCHITECTURES if RUN_MODE == "train" else ()
RUN_SMOKE_TEST = RUN_MODE == "smoke"
ALLOW_FINAL_TEST = os.environ.get("ALLOW_FINAL_TEST", "0") == "1"
REGENERATE_SPLIT = os.environ.get("REGENERATE_SPLIT", "0") == "1"

print("Racine projet :", PROJECT_ROOT)
print("Dataset       :", DATASET_DIR)
print("Artefacts     :", ARTIFACT_ROOT)
print("Device        :", DEVICE)
print("PyTorch       :", torch.__version__)
print("SMP           :", smp.__version__)
print("Mode          :", RUN_MODE)
if DEVICE.type == "cuda":
    print("GPU           :", torch.cuda.get_device_name(0))
print("Entraînements :", RUN_ARCHITECTURES or "aucun (mode sûr)")


Racine projet : /__modal/volumes/vo-88iL1FT1KqFvfccQkeg6aY/project
Dataset       : /__modal/volumes/vo-88iL1FT1KqFvfccQkeg6aY/project/water_v2
Artefacts     : /__modal/volumes/vo-88iL1FT1KqFvfccQkeg6aY/project/runs/unetplusplus
Device        : cuda
PyTorch       : 2.12.1+cu126
SMP           : 0.5.0
Mode          : train
GPU           : NVIDIA L4
Entraînements : ('unetplusplus',)


In [24]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False


def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


seed_everything(CONFIG.seed)


## 2. Audit des paires et statistiques de groupe

Le cache contient uniquement des chemins relatifs et des statistiques Water-v2. Les images et masques bruts restent hors Git. Un hash exact et un dHash visuel permettent de signaler les doublons traversant plusieurs groupes avant de figer le split.

In [25]:
SUPPORTED_SUFFIXES = {".jpg", ".jpeg", ".png"}


def group_of_pair(pair: tuple[Path, Path]) -> str:
    return pair[0].relative_to(DATASET_DIR / "JPEGImages").parts[0]


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def difference_hash(path: Path, size: int = 8) -> str:
    with Image.open(path) as image:
        gray = image.convert("L").resize((size + 1, size), Image.Resampling.BILINEAR)
        values = np.asarray(gray)
    bits = values[:, 1:] > values[:, :-1]
    return f"{int(''.join('1' if bit else '0' for bit in bits.flat), 2):016x}"


def audit_pairs(pairs: list[tuple[Path, Path]], cache_path: Path) -> pd.DataFrame:
    if cache_path.exists():
        cached = pd.read_csv(cache_path)
        if len(cached) == len(pairs):
            print(f"Cache réutilisé : {cache_path}")
            return cached

    rows: list[dict[str, Any]] = []
    for index, (image_path, mask_path) in enumerate(pairs, start=1):
        with Image.open(image_path) as image, Image.open(mask_path) as mask:
            image_size = image.size
            mask_size = mask.size
            if image_size != mask_size:
                raise ValueError(f"Désalignement {image_path}: {image_size} != {mask_size}")
            mask_array = np.asarray(mask.convert("L")) > 0

        rows.append(
            {
                "image": image_path.relative_to(DATASET_DIR).as_posix(),
                "mask": mask_path.relative_to(DATASET_DIR).as_posix(),
                "group": group_of_pair((image_path, mask_path)),
                "width": image_size[0],
                "height": image_size[1],
                "water_pixels": int(mask_array.sum()),
                "total_pixels": int(mask_array.size),
                "water_ratio": float(mask_array.mean()),
                "empty_mask": bool(not mask_array.any()),
                "image_sha256": sha256_file(image_path),
                "image_dhash": difference_hash(image_path),
            }
        )
        if index % 250 == 0:
            print(f"Audit : {index}/{len(pairs)}")

    result = pd.DataFrame(rows)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(cache_path, index=False)
    return result


def cross_group_duplicates(stats: pd.DataFrame, column: str) -> pd.DataFrame:
    counts = stats.groupby(column)["group"].nunique()
    duplicate_keys = counts[counts > 1].index
    return stats[stats[column].isin(duplicate_keys)].sort_values([column, "group", "image"])


In [26]:
pairs = find_water_v2_pairs(DATASET_DIR)
if not pairs:
    raise FileNotFoundError(f"Aucune paire Water-v2 trouvée sous {DATASET_DIR}")

pair_stats = audit_pairs(pairs, STATS_CACHE)
group_stats = (
    pair_stats.groupby("group", as_index=False)
    .agg(
        pair_count=("image", "size"),
        water_pixels=("water_pixels", "sum"),
        total_pixels=("total_pixels", "sum"),
        mean_image_water_ratio=("water_ratio", "mean"),
        empty_masks=("empty_mask", "sum"),
    )
)
group_stats["global_water_ratio"] = group_stats["water_pixels"] / group_stats["total_pixels"]
group_stats["empty_mask_ratio"] = group_stats["empty_masks"] / group_stats["pair_count"]
group_stats = group_stats.sort_values("pair_count", ascending=False).reset_index(drop=True)

exact_duplicates = cross_group_duplicates(pair_stats, "image_sha256")
visual_duplicates = cross_group_duplicates(pair_stats, "image_dhash")
print(f"Paires : {len(pair_stats):,} | Groupes : {len(group_stats)}")
print(f"Doublons exacts intergroupes : {exact_duplicates['image_sha256'].nunique()}")
print(f"dHash identiques intergroupes : {visual_duplicates['image_dhash'].nunique()}")
display(group_stats)


Cache réutilisé : /__modal/volumes/vo-88iL1FT1KqFvfccQkeg6aY/project/runs/unetplusplus/water_v2_pair_stats.csv
Paires : 2,400 | Groupes : 22
Doublons exacts intergroupes : 0
dHash identiques intergroupes : 0


,group,pair_count,water_pixels,total_pixels,mean_image_water_ratio,empty_masks,global_water_ratio,empty_mask_ratio
0,ADE20K,1888,144857741,712613223,0.223971,8,0.203277,0.004237
1,river_segs,300,59604946,160291445,0.395790,0,0.371854,0.000000
2,stream3_small,28,61056974,80640000,0.757155,0,0.757155,0.000000
3,buffalo0_small,28,15685782,58060800,0.270161,0,0.270161,0.000000
4,boston_harbor2_small_rois,27,2871697,6750000,0.425437,0,0.425437,0.000000
5,stream1,24,9901863,49766400,0.198967,0,0.198967,0.000000
6,stream0,15,5985090,31104000,0.192422,0,0.192422,0.000000
7,canal0,10,9296178,20736000,0.448311,0,0.448311,0.000000
8,gulf_crest,7,2522181,28600320,0.088187,0,0.088187,0.000000
9,aberlour,6,724195,1843200,0.392901,0,0.392901,0.000000


## 3. Split fixe 14 / 4 / 4

Le manifeste n'est pas écrasé silencieusement. Pour le recalculer intentionnellement, définir `REGENERATE_SPLIT=1`. Les doublons exacts intergroupes bloquent la génération ; les dHash identiques sont affichés pour vérification humaine.

In [27]:
FORCED_TRAIN_GROUPS = {"ADE20K", "river_segs"}


def partition_summary(stats: pd.DataFrame, groups: set[str]) -> dict[str, float | int]:
    subset = stats[stats["group"].isin(groups)]
    pair_count = int(subset["pair_count"].sum())
    total_pixels = int(subset["total_pixels"].sum())
    water_pixels = int(subset["water_pixels"].sum())
    empty_masks = int(subset["empty_masks"].sum())
    return {
        "groups": len(groups),
        "pairs": pair_count,
        "water_ratio": water_pixels / max(total_pixels, 1),
        "empty_mask_ratio": empty_masks / max(pair_count, 1),
    }


def split_score(stats: pd.DataFrame, validation: set[str], test: set[str]) -> float:
    remaining = stats[~stats["group"].isin(FORCED_TRAIN_GROUPS)]
    overall_water = remaining["water_pixels"].sum() / remaining["total_pixels"].sum()
    overall_empty = remaining["empty_masks"].sum() / remaining["pair_count"].sum()
    val = partition_summary(stats, validation)
    tst = partition_summary(stats, test)
    pair_scale = max(val["pairs"] + tst["pairs"], 1)
    return float(
        abs(val["pairs"] - tst["pairs"]) / pair_scale
        + abs(val["water_ratio"] - tst["water_ratio"])
        + 0.5 * abs(val["water_ratio"] - overall_water)
        + 0.5 * abs(tst["water_ratio"] - overall_water)
        + abs(val["empty_mask_ratio"] - tst["empty_mask_ratio"])
        + 0.5 * abs(val["empty_mask_ratio"] - overall_empty)
        + 0.5 * abs(tst["empty_mask_ratio"] - overall_empty)
    )


def generate_group_split(
    stats: pd.DataFrame,
    seed: int = 42,
    attempts: int = 100_000,
) -> dict[str, Any]:
    all_groups = set(stats["group"])
    missing = FORCED_TRAIN_GROUPS - all_groups
    if missing:
        raise ValueError(f"Groupes forcés absents : {sorted(missing)}")
    if not exact_duplicates.empty:
        raise ValueError(
            "Des doublons exacts relient plusieurs groupes. Inspectez `exact_duplicates` "
            "et regroupez leurs origines avant de figer le split."
        )

    candidates = sorted(all_groups - FORCED_TRAIN_GROUPS)
    if len(candidates) != 20:
        raise ValueError(f"22 groupes attendus, {len(all_groups)} trouvés")

    rng = np.random.default_rng(seed)
    best: tuple[float, tuple[str, ...], tuple[str, ...]] | None = None
    for _ in range(attempts):
        shuffled = rng.permutation(candidates)
        validation = tuple(sorted(shuffled[:4]))
        test = tuple(sorted(shuffled[4:8]))
        score = split_score(stats, set(validation), set(test))
        candidate = (score, validation, test)
        if best is None or candidate < best:
            best = candidate

    assert best is not None
    _, validation, test = best
    validation_groups = set(validation)
    test_groups = set(test)
    train_groups = all_groups - validation_groups - test_groups
    if len(train_groups) != 14 or len(validation_groups) != 4 or len(test_groups) != 4:
        raise AssertionError("Le split ne respecte pas 14/4/4")

    manifest: dict[str, Any] = {
        "version": 1,
        "dataset": "Water-v2",
        "seed": seed,
        "algorithm": "random_search_100000_balanced_groups_v1",
        "forced_train_groups": sorted(FORCED_TRAIN_GROUPS),
        "train_groups": sorted(train_groups),
        "validation_groups": sorted(validation_groups),
        "test_groups": sorted(test_groups),
        "summaries": {
            "train": partition_summary(stats, train_groups),
            "validation": partition_summary(stats, validation_groups),
            "test": partition_summary(stats, test_groups),
        },
    }
    canonical = json.dumps(manifest, sort_keys=True, separators=(",", ":"))
    manifest["split_hash"] = hashlib.sha256(canonical.encode("utf-8")).hexdigest()
    return manifest


def load_or_create_manifest(path: Path, regenerate: bool = False) -> dict[str, Any]:
    generated = generate_group_split(group_stats, seed=CONFIG.seed)
    if path.exists() and not regenerate:
        existing = json.loads(path.read_text(encoding="utf-8"))
        if existing.get("split_hash") != generated["split_hash"]:
            raise RuntimeError(
                "Le manifeste existant diffère du split déterministe recalculé. "
                "Ne l'écrasez qu'après un audit intentionnel."
            )
        return existing
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(generated, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    return generated


In [28]:
split_manifest = load_or_create_manifest(SPLIT_PATH, regenerate=REGENERATE_SPLIT)
train_groups = set(split_manifest["train_groups"])
validation_groups = set(split_manifest["validation_groups"])
test_groups = set(split_manifest["test_groups"])

train_pairs = [pair for pair in pairs if group_of_pair(pair) in train_groups]
validation_pairs = [pair for pair in pairs if group_of_pair(pair) in validation_groups]
test_pairs = [pair for pair in pairs if group_of_pair(pair) in test_groups]

assert train_groups.isdisjoint(validation_groups)
assert train_groups.isdisjoint(test_groups)
assert validation_groups.isdisjoint(test_groups)
assert len(train_pairs) + len(validation_pairs) + len(test_pairs) == len(pairs)

print("Split hash :", split_manifest["split_hash"])
display(pd.DataFrame(split_manifest["summaries"]).T)
print("Train      :", sorted(train_groups))
print("Validation :", sorted(validation_groups))
print("Test fermé :", sorted(test_groups))


Split hash : 5b85dc03c3871f4711cad91fcab5525d39825a2f06123add1b81ccbd3e0ffff5


,groups,pairs,water_ratio,empty_mask_ratio
train,14.0,2352.0,0.273079,0.003401
validation,4.0,24.0,0.395605,0.000000
test,4.0,24.0,0.397276,0.000000


Train      : ['ADE20K', 'boston_harbor2_small_rois', 'buffalo0_small', 'canal0', 'dublin', 'evesham-lock', 'galway-city', 'gulf_crest', 'mexico_beach_clip0', 'river_segs', 'stream0', 'stream1', 'stream2', 'stream3_small']
Validation : ['holiday_inn_clip0', 'holmrook', 'keswick_greta', 'worcester']
Test fermé : ['aberlour', 'auldgirth', 'bewdley', 'cockermouth']


## 4. Dataset, padding et augmentations

Les transformations géométriques sont appliquées de façon identique à l'image, au masque et au masque de pixels valides. Les pixels de letterboxing sont ainsi exclus de la loss et de toutes les métriques.

In [29]:
IMAGENET_MEAN = torch.tensor((0.485, 0.456, 0.406), dtype=torch.float32).view(3, 1, 1)
IMAGENET_STD = torch.tensor((0.229, 0.224, 0.225), dtype=torch.float32).view(3, 1, 1)


class WaterSegmentationDataset(Dataset):
    def __init__(
        self,
        pairs: list[tuple[Path, Path]],
        image_size: tuple[int, int],
        augment: bool = False,
    ) -> None:
        self.pairs = list(pairs)
        self.image_size = image_size
        self.augment = augment

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, index: int) -> dict[str, Any]:
        image_path, mask_path = self.pairs[index]
        image, mask, metadata = load_pair_with_padding(
            image_path,
            mask_path,
            size=self.image_size,
            return_metadata=True,
        )
        image_tensor = torch.from_numpy(image.copy()).permute(2, 0, 1)
        mask_tensor = torch.from_numpy(mask.copy()).unsqueeze(0).float()
        valid_tensor = torch.zeros_like(mask_tensor)
        pad_left, pad_top, pad_right, pad_bottom = metadata["padding"]
        height, width = mask_tensor.shape[-2:]
        valid_tensor[:, pad_top : height - pad_bottom, pad_left : width - pad_right] = 1.0

        if self.augment:
            if random.random() < 0.5:
                image_tensor = TF.hflip(image_tensor)
                mask_tensor = TF.hflip(mask_tensor)
                valid_tensor = TF.hflip(valid_tensor)

            if random.random() < 0.35:
                angle = random.uniform(-5.0, 5.0)
                max_dx = round(0.02 * width)
                max_dy = round(0.02 * height)
                translate = (random.randint(-max_dx, max_dx), random.randint(-max_dy, max_dy))
                scale = random.uniform(0.95, 1.05)
                image_tensor = TF.affine(
                    image_tensor, angle, translate, scale, 0.0,
                    interpolation=InterpolationMode.BILINEAR, fill=0.0,
                )
                mask_tensor = TF.affine(
                    mask_tensor, angle, translate, scale, 0.0,
                    interpolation=InterpolationMode.NEAREST, fill=0.0,
                )
                valid_tensor = TF.affine(
                    valid_tensor, angle, translate, scale, 0.0,
                    interpolation=InterpolationMode.NEAREST, fill=0.0,
                )

            if random.random() < 0.35:
                image_tensor = TF.adjust_brightness(image_tensor, random.uniform(0.85, 1.15))
                image_tensor = TF.adjust_contrast(image_tensor, random.uniform(0.85, 1.15))
                image_tensor = image_tensor.clamp(0.0, 1.0)

        image_tensor = (image_tensor - IMAGENET_MEAN) / IMAGENET_STD
        return {
            "image": image_tensor,
            "mask": (mask_tensor >= 0.5).float(),
            "valid": (valid_tensor >= 0.5).float(),
            "group": group_of_pair((image_path, mask_path)),
            "image_path": image_path.as_posix(),
        }


def create_loaders(config: TrainingConfig) -> tuple[DataLoader, DataLoader, DataLoader]:
    generator = torch.Generator().manual_seed(config.seed)
    common = {
        "batch_size": config.batch_size,
        "num_workers": config.num_workers,
        "pin_memory": DEVICE.type == "cuda",
        "worker_init_fn": seed_worker,
        "generator": generator,
        "persistent_workers": config.num_workers > 0,
    }
    train_loader = DataLoader(
        WaterSegmentationDataset(train_pairs, config.image_size, augment=True),
        shuffle=True,
        **common,
    )
    validation_loader = DataLoader(
        WaterSegmentationDataset(validation_pairs, config.image_size, augment=False),
        shuffle=False,
        **common,
    )
    test_loader = DataLoader(
        WaterSegmentationDataset(test_pairs, config.image_size, augment=False),
        shuffle=False,
        **common,
    )
    return train_loader, validation_loader, test_loader


train_loader, validation_loader, test_loader = create_loaders(CONFIG)
sample = next(iter(validation_loader))
print("Image :", tuple(sample["image"].shape), sample["image"].dtype)
print("Masque:", tuple(sample["mask"].shape), torch.unique(sample["mask"]))
print("Valide:", tuple(sample["valid"].shape), torch.unique(sample["valid"]))


Image : (8, 3, 512, 512) torch.float32
Masque: (8, 1, 512, 512) tensor([0., 1.])
Valide: (8, 1, 512, 512) tensor([0., 1.])


## 5. Modèles comparés

`activation=None` conserve les logits requis par `BCEWithLogitsLoss`. La deep supervision n'est pas activée dans cette première comparaison.

In [30]:
Architecture = Literal["unet", "unetplusplus"]


def build_model(architecture: Architecture, pretrained: bool = True) -> nn.Module:
    model_class = {
        "unet": smp.Unet,
        "unetplusplus": smp.UnetPlusPlus,
    }.get(architecture)
    if model_class is None:
        raise ValueError(f"Architecture inconnue : {architecture}")
    return model_class(
        encoder_name="resnet18",
        encoder_weights="imagenet" if pretrained else None,
        in_channels=3,
        classes=1,
        activation=None,
    )


def set_encoder_trainable(model: nn.Module, trainable: bool) -> None:
    for parameter in model.encoder.parameters():
        parameter.requires_grad = trainable


def keep_encoder_batch_norm_in_eval(model: nn.Module) -> None:
    for module in model.encoder.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()


def parameter_counts(model: nn.Module) -> dict[str, int]:
    return {
        "total": sum(parameter.numel() for parameter in model.parameters()),
        "trainable": sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad),
    }


for architecture in ("unet", "unetplusplus"):
    probe = build_model(architecture, pretrained=False)
    with torch.inference_mode():
        output = probe(torch.zeros(1, 3, 128, 128))
    print(architecture, parameter_counts(probe), "sortie", tuple(output.shape))
    del probe


unet {'total': 14328209, 'trainable': 14328209} sortie (1, 1, 128, 128)
unetplusplus {'total': 15970449, 'trainable': 15970449} sortie (1, 1, 128, 128)


## 6. Loss et métriques sans padding

Les scores globaux proviennent des comptes TP/FP/FN/TN cumulés. Les scores par image et par groupe sont également conservés. Deux masques vides obtiennent IoU/Dice = 1, conformément à la convention actuelle du projet.

In [31]:
EPSILON = 1e-7


def masked_bce_dice_loss(logits: torch.Tensor, targets: torch.Tensor, valid: torch.Tensor) -> torch.Tensor:
    valid_pixels = valid.sum().clamp_min(1.0)
    bce = (F.binary_cross_entropy_with_logits(logits, targets, reduction="none") * valid).sum() / valid_pixels
    probabilities = torch.sigmoid(logits.float())
    dimensions = (1, 2, 3)
    intersection = (probabilities * targets * valid).sum(dim=dimensions)
    denominator = ((probabilities + targets) * valid).sum(dim=dimensions)
    dice = (2.0 * intersection + 1.0) / (denominator + 1.0)
    return 0.5 * bce + 0.5 * (1.0 - dice.mean())


def confusion_counts(
    probabilities: torch.Tensor,
    targets: torch.Tensor,
    valid: torch.Tensor,
    threshold: float,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    predicted = probabilities >= threshold
    truth = targets >= 0.5
    selected = valid >= 0.5
    dims = (1, 2, 3)
    tp = (predicted & truth & selected).sum(dim=dims)
    fp = (predicted & ~truth & selected).sum(dim=dims)
    fn = (~predicted & truth & selected).sum(dim=dims)
    tn = (~predicted & ~truth & selected).sum(dim=dims)
    return tp, fp, fn, tn


def metrics_from_counts(tp: float, fp: float, fn: float, tn: float) -> dict[str, float]:
    union = tp + fp + fn
    predicted_and_true = 2.0 * tp + fp + fn
    return {
        "iou": (tp + EPSILON) / (union + EPSILON),
        "dice": (2.0 * tp + EPSILON) / (predicted_and_true + EPSILON),
        "precision": (tp + EPSILON) / (tp + fp + EPSILON),
        "recall": (tp + EPSILON) / (tp + fn + EPSILON),
        "accuracy": (tp + tn + EPSILON) / (tp + fp + fn + tn + EPSILON),
    }


def binary_boundary(mask: np.ndarray) -> np.ndarray:
    tensor = torch.from_numpy(mask.astype(np.float32))[None, None]
    eroded = -F.max_pool2d(-tensor, kernel_size=3, stride=1, padding=1)
    return (tensor - eroded).squeeze().numpy() > 0


def boundary_f1_score(truth: np.ndarray, prediction: np.ndarray, tolerance: int = 2) -> float:
    truth_boundary = binary_boundary(truth)
    prediction_boundary = binary_boundary(prediction)
    if not truth_boundary.any() and not prediction_boundary.any():
        return 1.0
    kernel = 2 * tolerance + 1
    truth_dilated = F.max_pool2d(
        torch.from_numpy(truth_boundary.astype(np.float32))[None, None], kernel, 1, tolerance
    ).squeeze().numpy() > 0
    prediction_dilated = F.max_pool2d(
        torch.from_numpy(prediction_boundary.astype(np.float32))[None, None], kernel, 1, tolerance
    ).squeeze().numpy() > 0
    precision = (prediction_boundary & truth_dilated).sum() / max(prediction_boundary.sum(), 1)
    recall = (truth_boundary & prediction_dilated).sum() / max(truth_boundary.sum(), 1)
    return float((2 * precision * recall + EPSILON) / (precision + recall + EPSILON))


## 7. Boucle d'entraînement, reprise et sélection

Le meilleur checkpoint est sélectionné uniquement sur l'IoU eau de validation au seuil 0,5. `last.pt` permet de reprendre après une interruption Modal.

In [32]:
def make_optimizer(model: nn.Module, config: TrainingConfig) -> torch.optim.Optimizer:
    encoder_ids = {id(parameter) for parameter in model.encoder.parameters()}
    decoder_parameters = [parameter for parameter in model.parameters() if id(parameter) not in encoder_ids]
    return torch.optim.AdamW(
        [
            {"params": model.encoder.parameters(), "lr": config.encoder_learning_rate},
            {"params": decoder_parameters, "lr": config.decoder_learning_rate},
        ],
        weight_decay=config.weight_decay,
    )


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scaler: torch.amp.GradScaler,
    config: TrainingConfig,
) -> float:
    model.train()
    keep_encoder_batch_norm_in_eval(model)
    optimizer.zero_grad(set_to_none=True)
    total_loss = 0.0
    sample_count = 0

    for batch_index, batch in enumerate(loader):
        images = batch["image"].to(DEVICE, non_blocking=True)
        masks = batch["mask"].to(DEVICE, non_blocking=True)
        valid = batch["valid"].to(DEVICE, non_blocking=True)
        with torch.amp.autocast(device_type=DEVICE.type, enabled=config.use_amp and DEVICE.type == "cuda"):
            logits = model(images)
            loss = masked_bce_dice_loss(logits, masks, valid)

        scaler.scale(loss / config.gradient_accumulation_steps).backward()
        should_step = (
            (batch_index + 1) % config.gradient_accumulation_steps == 0
            or batch_index + 1 == len(loader)
        )
        if should_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        batch_size = images.shape[0]
        total_loss += loss.item() * batch_size
        sample_count += batch_size
    return total_loss / max(sample_count, 1)


@torch.inference_mode()
def collect_predictions(
    model: nn.Module,
    loader: DataLoader,
    config: TrainingConfig,
) -> tuple[float, list[dict[str, Any]]]:
    model.eval()
    total_loss = 0.0
    sample_count = 0
    records: list[dict[str, Any]] = []
    for batch in loader:
        images = batch["image"].to(DEVICE, non_blocking=True)
        masks = batch["mask"].to(DEVICE, non_blocking=True)
        valid = batch["valid"].to(DEVICE, non_blocking=True)
        with torch.amp.autocast(device_type=DEVICE.type, enabled=config.use_amp and DEVICE.type == "cuda"):
            logits = model(images)
            loss = masked_bce_dice_loss(logits, masks, valid)
        probabilities = torch.sigmoid(logits.float()).cpu().numpy()
        masks_np = masks.cpu().numpy()
        valid_np = valid.cpu().numpy()
        for index in range(images.shape[0]):
            records.append(
                {
                    "probability": probabilities[index, 0],
                    "truth": masks_np[index, 0] >= 0.5,
                    "valid": valid_np[index, 0] >= 0.5,
                    "group": batch["group"][index],
                    "image_path": batch["image_path"][index],
                }
            )
        total_loss += loss.item() * images.shape[0]
        sample_count += images.shape[0]
    return total_loss / max(sample_count, 1), records


def summarize_predictions(
    records: list[dict[str, Any]],
    threshold: float,
    boundary_tolerance: int,
) -> tuple[dict[str, float], pd.DataFrame, pd.DataFrame]:
    rows: list[dict[str, Any]] = []
    global_counts = np.zeros(4, dtype=np.float64)
    for record in records:
        valid = record["valid"]
        truth = record["truth"] & valid
        prediction = (record["probability"] >= threshold) & valid
        tp = float((prediction & truth).sum())
        fp = float((prediction & ~truth & valid).sum())
        fn = float((~prediction & truth & valid).sum())
        tn = float((~prediction & ~truth & valid).sum())
        global_counts += (tp, fp, fn, tn)
        row = {
            "image_path": record["image_path"],
            "group": record["group"],
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "tn": tn,
            **metrics_from_counts(tp, fp, fn, tn),
            "boundary_f1": boundary_f1_score(truth, prediction, boundary_tolerance),
        }
        rows.append(row)

    per_image = pd.DataFrame(rows)
    per_group_counts = per_image.groupby("group", as_index=False)[["tp", "fp", "fn", "tn"]].sum()
    metric_rows = []
    for _, row in per_group_counts.iterrows():
        values = metrics_from_counts(row.tp, row.fp, row.fn, row.tn)
        values.update(
            {
                "group": row.group,
                "images": int((per_image["group"] == row.group).sum()),
                "boundary_f1": float(per_image.loc[per_image["group"] == row.group, "boundary_f1"].mean()),
            }
        )
        metric_rows.append(values)
    per_group = pd.DataFrame(metric_rows)
    global_metrics = metrics_from_counts(*global_counts)
    global_metrics["boundary_f1"] = float(per_image["boundary_f1"].mean())
    global_metrics["mean_image_iou"] = float(per_image["iou"].mean())
    global_metrics["mean_image_dice"] = float(per_image["dice"].mean())
    global_metrics.update(dict(zip(("tp", "fp", "fn", "tn"), global_counts.tolist())))
    return global_metrics, per_image, per_group


def run_directory(architecture: Architecture, seed: int) -> Path:
    return ARTIFACT_ROOT / f"{architecture}_resnet18_seed{seed}_{split_manifest['split_hash'][:8]}"


def save_checkpoint(
    path: Path,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler.ReduceLROnPlateau,
    scaler: torch.amp.GradScaler,
    architecture: Architecture,
    config: TrainingConfig,
    epoch: int,
    best_iou: float,
    history: list[dict[str, float]],
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "architecture": architecture,
            "encoder_name": "resnet18",
            "encoder_weights": "imagenet",
            "state_dict": getattr(model, '_orig_mod', model).state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "config": asdict(config),
            "split_hash": split_manifest["split_hash"],
            "epoch": epoch,
            "best_validation_iou": best_iou,
            "history": history,
        },
        path,
    )


def fit(architecture: Architecture, config: TrainingConfig, resume: bool = True) -> dict[str, Any]:
    if DEVICE.type != "cuda" and not RUN_SMOKE_TEST:
        raise RuntimeError("CUDA est requis pour l'entraînement complet. Utilisez Modal L4 ou activez le smoke test.")
    seed_everything(config.seed)
    output_dir = run_directory(architecture, config.seed)
    output_dir.mkdir(parents=True, exist_ok=True)
    model = build_model(architecture, pretrained=True).to(DEVICE)
    model = model.to(memory_format=torch.channels_last)
    model = torch.compile(model, mode="default")
    set_encoder_trainable(model, False)
    optimizer = make_optimizer(model, config)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)
    scaler = torch.amp.GradScaler("cuda", enabled=config.use_amp and DEVICE.type == "cuda")
    best_path = output_dir / "best.pt"
    last_path = output_dir / "last.pt"
    history: list[dict[str, float]] = []
    best_iou = -math.inf
    start_epoch = 1

    if resume and last_path.exists():
        checkpoint = torch.load(last_path, map_location=DEVICE, weights_only=False)
        if checkpoint["split_hash"] != split_manifest["split_hash"]:
            raise RuntimeError("Le checkpoint utilise un autre split")
        model.load_state_dict(checkpoint["state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        scaler.load_state_dict(checkpoint["scaler_state_dict"])
        history = checkpoint["history"]
        best_iou = float(checkpoint["best_validation_iou"])
        start_epoch = int(checkpoint["epoch"]) + 1
        set_encoder_trainable(model, start_epoch > config.freeze_encoder_epochs)
        print(f"Reprise à l'epoch {start_epoch}")

    train_loader, validation_loader, _ = create_loaders(config)
    patience = 0
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    print(f"Début entraînement | batches/epoch: {len(train_loader)} | compile: torch.compile actif")

    for epoch in range(start_epoch, config.max_epochs + 1):
        if epoch == config.freeze_encoder_epochs + 1:
            set_encoder_trainable(model, True)
            print("Encodeur ResNet18 dégelé")
        train_loss = train_one_epoch(model, train_loader, optimizer, scaler, config)
        validation_loss, validation_records = collect_predictions(model, validation_loader, config)
        validation_metrics, _, _ = summarize_predictions(
            validation_records, config.threshold, config.boundary_tolerance
        )
        scheduler.step(validation_metrics["iou"])
        row = {
            "epoch": float(epoch),
            "train_loss": train_loss,
            "validation_loss": validation_loss,
            **{f"validation_{key}": float(value) for key, value in validation_metrics.items()},
            "encoder_lr": optimizer.param_groups[0]["lr"],
            "decoder_lr": optimizer.param_groups[1]["lr"],
        }
        history.append(row)
        improved = validation_metrics["iou"] > best_iou
        if improved:
            best_iou = validation_metrics["iou"]
            patience = 0
            save_checkpoint(best_path, model, optimizer, scheduler, scaler, architecture, config, epoch, best_iou, history)
        else:
            patience += 1
        save_checkpoint(last_path, model, optimizer, scheduler, scaler, architecture, config, epoch, best_iou, history)
        pd.DataFrame(history).to_csv(output_dir / "history.csv", index=False)
        print(
            f"Epoch {epoch:03d} | loss {train_loss:.4f}/{validation_loss:.4f} | "
            f"IoU {validation_metrics['iou']:.4f} | Dice {validation_metrics['dice']:.4f} | "
            f"Boundary F1 {validation_metrics['boundary_f1']:.4f}"
        )
        if patience >= config.early_stopping_patience:
            print("Early stopping")
            break

    checkpoint = torch.load(best_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["state_dict"])
    validation_loss, validation_records = collect_predictions(model, validation_loader, config)
    metrics, per_image, per_group = summarize_predictions(
        validation_records, config.threshold, config.boundary_tolerance
    )
    metrics.update(
        {
            "loss": validation_loss,
            "best_epoch": checkpoint["epoch"],
            "duration_seconds": time.perf_counter() - started,
            "max_vram_mb": torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else 0.0,
            "parameter_count": parameter_counts(model)["total"],
        }
    )
    (output_dir / "validation_metrics.json").write_text(
        json.dumps(metrics, indent=2), encoding="utf-8"
    )
    per_image.to_csv(output_dir / "validation_per_image.csv", index=False)
    per_group.to_csv(output_dir / "validation_per_group.csv", index=False)
    (output_dir / "config.json").write_text(
        json.dumps({"architecture": architecture, "config": asdict(config), "split": split_manifest}, indent=2),
        encoding="utf-8",
    )
    return {"model": model, "records": validation_records, "metrics": metrics, "output_dir": output_dir}


## 8. Préflight mémoire et smoke test

Le préflight tente le batch 32 en premier. En cas d'OOM, il divise par 2 et compense par accumulation jusqu'à trouver une configuration stable. Le modèle bénéficie de `torch.compile`, du format `channels_last` et de `cudnn.benchmark` pour saturer le GPU L4.

In [33]:
def preflight_config(config: TrainingConfig, architecture: Architecture = "unetplusplus") -> TrainingConfig:
    candidate = config
    while True:
        model = build_model(architecture, pretrained=False).to(DEVICE)
        try:
            images = torch.zeros(candidate.batch_size, 3, *reversed(candidate.image_size), device=DEVICE)
            masks = torch.zeros(candidate.batch_size, 1, *reversed(candidate.image_size), device=DEVICE)
            valid = torch.ones_like(masks)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=candidate.use_amp and DEVICE.type == "cuda"):
                loss = masked_bce_dice_loss(model(images), masks, valid)
            loss.backward()
            print(
                f"Préflight réussi : batch={candidate.batch_size}, "
                f"accumulation={candidate.gradient_accumulation_steps}"
            )
            return candidate
        except torch.cuda.OutOfMemoryError:
            if candidate.batch_size == 1:
                raise
            fallback_batch_size = max(1, candidate.batch_size // 2)
            fallback_accumulation = candidate.gradient_accumulation_steps * (
                candidate.batch_size // fallback_batch_size
            )
            print(
                f"OOM avec batch {candidate.batch_size} : repli vers batch "
                f"{fallback_batch_size} / accumulation {fallback_accumulation}"
            )
            candidate = replace(
                candidate,
                batch_size=fallback_batch_size,
                gradient_accumulation_steps=fallback_accumulation,
            )
        finally:
            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()


if RUN_SMOKE_TEST or RUN_ARCHITECTURES:
    CONFIG = preflight_config(CONFIG)

if RUN_SMOKE_TEST:
    smoke_config = replace(
        CONFIG,
        image_size=(128, 128),
        batch_size=1,
        gradient_accumulation_steps=1,
        max_epochs=2,
        freeze_encoder_epochs=1,
        early_stopping_patience=2,
        num_workers=0,
    )
    original_train_pairs = train_pairs
    original_validation_pairs = validation_pairs
    train_pairs = train_pairs[:8]
    validation_pairs = validation_pairs[:4]
    smoke_result = fit("unetplusplus", smoke_config, resume=False)
    train_pairs = original_train_pairs
    validation_pairs = original_validation_pairs
    print("Smoke test validé :", smoke_result["metrics"])


Préflight réussi : batch=8, accumulation=1


## 9. Entraînements comparatifs

Pour une première campagne : `RUN_ARCHITECTURES=unet,unetplusplus`. Sur Modal, il est préférable de lancer les deux architectures séparément afin que chaque run possède ses logs et puisse reprendre indépendamment.

In [ ]:
experiment_results: dict[str, dict[str, Any]] = {}
for architecture in RUN_ARCHITECTURES:
    if architecture not in {"unet", "unetplusplus"}:
        raise ValueError(f"RUN_ARCHITECTURES contient une valeur invalide : {architecture}")
    experiment_results[architecture] = fit(architecture, CONFIG, resume=True)


Reprise à l'epoch 3
Epoch 003 | loss 0.4051/0.1623 | IoU 0.8353 | Dice 0.9103 | Boundary F1 0.3224
Epoch 004 | loss 0.3047/0.1441 | IoU 0.8380 | Dice 0.9118 | Boundary F1 0.3744
Epoch 005 | loss 0.2754/0.1180 | IoU 0.8743 | Dice 0.9329 | Boundary F1 0.4718
Encodeur ResNet18 dégelé
Epoch 006 | loss 0.2870/0.1119 | IoU 0.8810 | Dice 0.9367 | Boundary F1 0.4837
Epoch 007 | loss 0.2665/0.1039 | IoU 0.8823 | Dice 0.9375 | Boundary F1 0.5051
Epoch 008 | loss 0.2435/0.1122 | IoU 0.8761 | Dice 0.9339 | Boundary F1 0.4679
Epoch 009 | loss 0.2300/0.1138 | IoU 0.8745 | Dice 0.9330 | Boundary F1 0.4948


In [ ]:
def available_validation_results(seed: int = 42) -> pd.DataFrame:
    rows = []
    for architecture in ("unet", "unetplusplus"):
        path = run_directory(architecture, seed) / "validation_metrics.json"
        if path.exists():
            row = json.loads(path.read_text(encoding="utf-8"))
            row["architecture"] = architecture
            rows.append(row)
    return pd.DataFrame(rows)


comparison = available_validation_results(CONFIG.seed)
if comparison.empty:
    print("Aucun résultat complet disponible pour le moment.")
else:
    columns = [
        "architecture", "iou", "dice", "precision", "recall", "boundary_f1",
        "mean_image_iou", "best_epoch", "duration_seconds", "max_vram_mb", "parameter_count",
    ]
    display(comparison[columns].sort_values("iou", ascending=False))
    comparison.set_index("architecture")[["iou", "dice", "boundary_f1"]].plot.bar(
        figsize=(9, 4), ylim=(0, 1), title="Validation — comparaison à seuil 0,5"
    )
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()


## 10. Réglage du seuil sur validation

Le seuil est recherché entre 0,20 et 0,80. Cette cellule ne consulte jamais le test.

In [ ]:
def tune_threshold(
    records: list[dict[str, Any]],
    boundary_tolerance: int,
) -> tuple[float, pd.DataFrame]:
    coarse_rows = []
    for threshold in np.arange(0.20, 0.801, 0.05):
        metrics, _, _ = summarize_predictions(records, float(threshold), boundary_tolerance)
        coarse_rows.append({"threshold": float(threshold), **metrics})
    coarse = pd.DataFrame(coarse_rows)
    coarse_best = float(coarse.loc[coarse["iou"].idxmax(), "threshold"])
    fine_rows = []
    for threshold in np.arange(max(0.05, coarse_best - 0.05), min(0.95, coarse_best + 0.05) + 0.001, 0.01):
        metrics, _, _ = summarize_predictions(records, float(threshold), boundary_tolerance)
        fine_rows.append({"threshold": round(float(threshold), 2), **metrics})
    fine = pd.DataFrame(fine_rows).drop_duplicates("threshold")
    best_threshold = float(fine.sort_values(["iou", "dice"], ascending=False).iloc[0]["threshold"])
    return best_threshold, pd.concat([coarse, fine], ignore_index=True).drop_duplicates("threshold").sort_values("threshold")


for architecture, result in experiment_results.items():
    best_threshold, threshold_table = tune_threshold(result["records"], CONFIG.boundary_tolerance)
    threshold_table.to_csv(result["output_dir"] / "validation_threshold_sweep.csv", index=False)
    selection = {"architecture": architecture, "threshold": best_threshold, "split_hash": split_manifest["split_hash"]}
    (result["output_dir"] / "selected_threshold.json").write_text(
        json.dumps(selection, indent=2), encoding="utf-8"
    )
    print(f"{architecture}: seuil de validation retenu = {best_threshold:.2f}")


## 11. Test final — verrou explicite

Cette cellule exige `ALLOW_FINAL_TEST=1`. Elle doit être exécutée une seule fois, après comparaison multi-seed et gel de l'architecture, de la loss, du prétraitement et du seuil.

In [ ]:
def evaluate_final_test(architecture: Architecture, seed: int = 42) -> dict[str, float]:
    if not ALLOW_FINAL_TEST:
        raise PermissionError("Test final verrouillé : définir explicitement ALLOW_FINAL_TEST=1")
    output_dir = run_directory(architecture, seed)
    threshold_info = json.loads((output_dir / "selected_threshold.json").read_text(encoding="utf-8"))
    checkpoint = torch.load(output_dir / "best.pt", map_location=DEVICE, weights_only=False)
    if checkpoint["split_hash"] != split_manifest["split_hash"]:
        raise RuntimeError("Split incompatible")
    model = build_model(architecture, pretrained=False).to(DEVICE)
    model.load_state_dict(checkpoint["state_dict"])
    _, _, loader = create_loaders(CONFIG)
    test_loss, records = collect_predictions(model, loader, CONFIG)
    metrics, per_image, per_group = summarize_predictions(
        records, threshold_info["threshold"], CONFIG.boundary_tolerance
    )
    metrics.update({"loss": test_loss, "threshold": threshold_info["threshold"]})
    (output_dir / "FINAL_test_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    per_image.to_csv(output_dir / "FINAL_test_per_image.csv", index=False)
    per_group.to_csv(output_dir / "FINAL_test_per_group.csv", index=False)
    return metrics


if ALLOW_FINAL_TEST:
    if len(RUN_ARCHITECTURES) != 1:
        raise RuntimeError("Indiquez une seule architecture finale dans RUN_ARCHITECTURES")
    final_metrics = evaluate_final_test(RUN_ARCHITECTURES[0], CONFIG.seed)
    display(pd.Series(final_metrics, name="test final"))
else:
    print("Test final fermé.")


## 12. Exécution directe dans Modal

Ce fichier est conçu pour être ouvert directement dans un kernel **L4** avec le Volume monté sous `/mnt/water-segmentation`.

1. Exécuter toutes les cellules une première fois avec `RUN_MODE = "smoke"` (valeur par défaut).
2. Vérifier que deux epochs terminent et que les fichiers apparaissent sous `runs/unetplusplus/`.
3. Définir ensuite la variable d'environnement `RUN_MODE` à `train`, redémarrer le kernel et réexécuter toutes les cellules.
4. En cas d'interruption, relancer en mode `train` : le notebook reprend depuis `last.pt`.

Le cache d'audit, le split, les historiques et les checkpoints sont enregistrés dans le Volume persistant. La cellule ci-dessous conserve le lanceur SDK uniquement pour compatibilité avec une exécution depuis un PC ; elle ne lance rien avec **Run all** dans Modal.

In [ ]:
# Compatibilité facultative pour un lancement depuis un environnement local.
# Dans un notebook Modal déjà ouvert sur L4, ne pas appeler ces fonctions.
def define_modal_app():
    try:
        import modal
    except ImportError as exc:
        raise ImportError("Installez le groupe dev avec `uv sync --group dev`") from exc

    notebook_path = PROJECT_ROOT / "notebooks" / "03_deep_learning_unet++.ipynb"
    image = (
        modal.Image.debian_slim(python_version="3.12")
        .pip_install(
            "torch==2.12.1",
            "torchvision==0.27.1",
            index_url="https://download.pytorch.org/whl/cu126",
        )
        .pip_install(
            "segmentation-models-pytorch==0.5.0",
            "numpy>=2.5.1",
            "pandas>=3.0.3",
            "pillow>=12.3.0",
            "matplotlib>=3.11.0",
            "opencv-python-headless>=4.11",
            "nbconvert>=7.16",
            "ipykernel>=7.3",
        )
        .add_local_dir(str(PROJECT_ROOT / "src"), remote_path="/workspace/src", copy=True)
        .add_local_file(str(PROJECT_ROOT / "pyproject.toml"), remote_path="/workspace/pyproject.toml", copy=True)
        .add_local_file(str(notebook_path), remote_path="/workspace/notebooks/03_deep_learning_unet++.ipynb", copy=True)
    )
    app = modal.App("water-unetplusplus-training", image=image)
    data_volume = modal.Volume.from_name("water-detection-data", create_if_missing=False)
    artifact_volume = modal.Volume.from_name("water-detection-artifacts", create_if_missing=True)

    @app.function(
        gpu="L4",
        timeout=6 * 60 * 60,
        volumes={"/data": data_volume, "/artifacts": artifact_volume},
    )
    def execute_notebook(architecture: str, seed: int = 42, smoke_test: bool = False) -> str:
        environment = os.environ.copy()
        environment.update(
            {
                "PROJECT_ROOT": "/workspace",
                "WATER_V2_DIR": "/data/water_v2",
                "ARTIFACT_DIR": "/artifacts/runs",
                "RUN_ARCHITECTURES": "" if smoke_test else architecture,
                "RUN_SMOKE_TEST": "1" if smoke_test else "0",
                "ALLOW_FINAL_TEST": "0",
                "PYTHONPATH": "/workspace/src",
                "PYTHONHASHSEED": str(seed),
            }
        )
        output_name = f"executed_{architecture}_seed{seed}.ipynb"
        output = f"/artifacts/{output_name}"
        subprocess.run(
            [
                sys.executable,
                "-m",
                "jupyter",
                "nbconvert",
                "--to",
                "notebook",
                "--execute",
                "/workspace/notebooks/03_deep_learning_unet++.ipynb",
                "--output",
                output_name,
                "--output-dir",
                "/artifacts",
                "--ExecutePreprocessor.timeout=-1",
            ],
            check=True,
            cwd="/workspace",
            env=environment,
        )
        artifact_volume.commit()
        return output

    return app, execute_notebook


def launch_on_modal(architecture: Architecture, seed: int = 42, smoke_test: bool = False) -> str:
    app, remote_function = define_modal_app()
    with app.run():
        return remote_function.remote(architecture, seed, smoke_test)


# Exemples à lancer volontairement depuis le kernel local :
# launch_on_modal("unetplusplus", smoke_test=True)
# launch_on_modal("unet", seed=42)
# launch_on_modal("unetplusplus", seed=42)

required_paths = {
    "pyproject": PROJECT_ROOT / "pyproject.toml",
    "sources": PROJECT_ROOT / "src" / "water_detection_methods",
    "images": DATASET_DIR / "JPEGImages",
    "masks": DATASET_DIR / "Annotations",
}
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
for label, path in required_paths.items():
    print(f"{label:10s} | {'OK' if path.exists() else 'ABSENT':6s} | {path}")
if not all(path.exists() for path in required_paths.values()):
    raise FileNotFoundError("Le Volume Modal ne contient pas toute la structure attendue")
if DEVICE.type != "cuda":
    raise RuntimeError("Sélectionnez un kernel Modal avec GPU L4 avant l'entraînement")


## 13. Critères de fin

Avant toute conclusion :

1. vérifier que le split contient toutes les paires sans recouvrement de groupes ;
2. réussir le smoke test et la reprise depuis `last.pt` ;
3. entraîner les deux architectures avec la seed 42 ;
4. si U-Net++ est compétitif, répéter les deux modèles avec les seeds 17, 42 et 73 ;
5. comparer moyenne, écart-type et résultats par groupe ;
6. figer un seuil commun sur validation ;
7. seulement ensuite ouvrir le test final.

U-Net++ est retenu uniquement s'il améliore IoU/Dice/Boundary F1 sans échec critique sur un groupe.